# Batch BO・非同期BO・Fantasization：実験の進捗に応じた候補提案

**同時に複数点を測るBatch BO**、**実験中の候補を考慮する非同期BO**、**仮想観測を使うFantasization**を区別して学びます。

BoTorch / robotorchanの既存公開APIを使い、`q=2` の同時提案、`X_pending`、`fantasize()`、実験完了後のGP再学習、条件付きEPIGの候補提案を実行します。独自の獲得関数やスケジューラは定義しません。


## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 3つの操作を区別する

- **Batch BO**：次の実験候補を一度に複数選びます（`q=2` など）。
- **非同期BO**：実験装置が空いた時点で、まだ完了していない実験を `X_pending` として考慮し、追加候補を提案します。
- **Fantasization**：実験結果がまだ不明な点に仮の応答をサンプリングし、仮想的に条件付けされたモデルを作ります。

これらは組み合わせて利用できますが、**同じ処理ではありません**。

## 2. Completed observations define the fitted model

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.linspace(0.05, 0.95, 9).unsqueeze(-1)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()
bounds = torch.tensor([[0.0], [1.0]])

## 3. Joint batch BO

`q=2` acquisitionは2候補をjoint decisionとして評価します。

### q-batchの同時提案

`qLogNoisyExpectedImprovement` は、現在の観測データ `X_baseline` とモデルの予測に基づいて改善を評価します。`q=2` なら2候補を**1組の意思決定**として最適化します。

返される `[2,1]` は「2候補 × 1入力変数」です。目的関数が2つあるという意味ではありません。

In [ ]:
batch_acq = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=101),
)
batch_candidate, batch_value = optimize_acqf(
    acq_function=batch_acq,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert batch_candidate.shape == torch.Size([2, 1])
print("joint q=2 candidate:", batch_candidate.squeeze(-1))
print("acquisition value:", batch_value)

## 4. Async BO uses X_pending as acquisition context

未完了の実験はtraining observationへ追加しません。`X_pending`としてacquisitionへ渡し、次候補がpending evaluationsを考慮するようにします。

### 未完了実験の入力を伝える

`X_pending` は測定開始済みで結果がまだ返っていない入力です。次の候補提案では、これらを無視して同じような条件を重複して測定しないように、獲得関数へ渡します。

`X_pending` に目的関数値 `Y` はありません。**未完了点を訓練データへ追加してGPを再学習してはいけません**。

In [ ]:
X_pending = torch.tensor([[0.32], [0.68]])
async_acq = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=202),
    X_pending=X_pending,
)
async_candidate, async_value = optimize_acqf(
    acq_function=async_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

assert async_candidate.shape == torch.Size([1, 1])
print("pending points:", X_pending.squeeze(-1))
print("next async candidate:", async_candidate.squeeze(-1))
print("async acquisition value:", async_value)

## 5. Pending points are not training observations

pending pointにはまだ`Y`がありません。したがって`train_X/train_Y`へ追加してrefitするのではなく、acquisition contextとして保持します。

In [ ]:
assert model.train_inputs[0].shape[0] == train_X.shape[0]
assert X_pending.shape[0] == 2
print("training rows:", train_X.shape[0])
print("pending rows supplied to acquisition:", X_pending.shape[0])

## 6. Model fantasization is a different operation

`fantasize()`はpending contextそのものではなく、指定したXで仮想Yをposteriorからsampleし、その仮想観測で条件付けしたmodel batchを構築します。

### Fantasizationの仕組み

`model.fantasize(X, sampler)` は指定した入力の**仮想的な観測値**を事後分布からサンプリングし、その値で条件付けしたモデルを生成します。

仮想観測の数が4なら4通りの仮説を持つモデルバッチができますが、実際の実験を4回実施したことにはなりません。

In [ ]:
fantasy_X = torch.tensor([[0.40]])
fantasy_sampler = SobolQMCNormalSampler(torch.Size([4]), seed=303)
fantasy_model = model.fantasize(
    X=fantasy_X,
    sampler=fantasy_sampler,
)

test_X = torch.tensor([[0.20], [0.80]])
with torch.no_grad():
    base_mean = model.posterior(test_X).mean
    fantasy_mean = fantasy_model.posterior(test_X).mean

print("base posterior mean shape:", base_mean.shape)
print("fantasy posterior mean shape:", fantasy_mean.shape)

## 7. Fantasy batch dimension is not q

fantasy sample数4はoptimizerの`q=4`を意味しません。fantasy-conditioned modelのbatch dimensionとcandidate q dimensionは独立です。

### Fantasyのバッチ軸と候補数qは異なる

`fantasy_mean.shape` の先頭の4は**仮想観測のシナリオ数**、末尾の `[2,1]` は**予測対象の2入力点 × 1出力**です。

最適化で新しく提案する候補数 `q` と、fantasyシナリオの数を混同しないようにします。

In [ ]:
assert fantasy_mean.shape[-2:] == torch.Size([2, 1])
assert fantasy_mean.shape[0] == 4
print("fantasy batch size:", fantasy_mean.shape[0])
print("posterior q at test_X:", fantasy_mean.shape[-2])

## 8. Completed pending evaluation becomes training data

実験が完了して`Y`が得られた時点で初めてtraining dataへ追加し、通常のmodel update/refitへ戻ります。

### 実験が完了したらGPを更新する

ここではpendingのうち最初の1点だけが完了したと仮定し、人工関数から実際の応答を取得します。完了した点は訓練データへ移し、残りのpending点は引き続き `X_pending` に保持します。

再学習後のGPを使って新しい獲得関数を作り直し、実験装置が次に測定する候補を提案します。

In [ ]:
completed_X = X_pending[:1]
completed_Y = response(completed_X)
updated_X = torch.cat([train_X, completed_X], dim=0)
updated_Y = torch.cat([train_Y, completed_Y], dim=0)

updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()

remaining_pending = X_pending[1:]
updated_async_acq = qLogNoisyExpectedImprovement(
    model=updated_model,
    X_baseline=updated_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=404),
    X_pending=remaining_pending,
)
next_candidate, _ = optimize_acqf(
    acq_function=updated_async_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

print("completed X:", completed_X.squeeze(-1))
print("remaining pending:", remaining_pending.squeeze(-1))
print("next candidate:", next_candidate.squeeze(-1))

### 完了した実験による予測更新を可視化する

完了済みの1点を追加した前後で、GPの予測平均と標準偏差を比較します。まだ未完了の点は**図に表示しても学習データには含めません**。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
with torch.no_grad():
    before_posterior = model.posterior(plot_X)
    after_posterior = updated_model.posterior(plot_X)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X.squeeze(-1), response(plot_X).squeeze(-1), label="True response")
for post, label in ((before_posterior, "Before"), (after_posterior, "After")):
    center = post.mean.squeeze(-1)
    std = post.variance.clamp_min(0).sqrt().squeeze(-1)
    ax.plot(plot_X.squeeze(-1), center, label=f"{label} mean")
    ax.fill_between(plot_X.squeeze(-1), center - 2 * std, center + 2 * std, alpha=0.12)
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), s=18, label="Initial observations")
ax.scatter(completed_X.squeeze(-1), completed_Y.squeeze(-1), marker="x", s=80, label="Completed observation")
ax.axvline(remaining_pending.item(), linestyle="--", label="Still pending")
ax.set(xlabel="x", ylabel="response", title="Posterior update after one pending experiment completes")
ax.legend(fontsize=8)
plt.show()

## 9. Four independent axes

| Axis | Meaning |
| --- | --- |
| external `q` | real candidates selected in this optimization call |
| `X_pending` rows | real evaluations already launched but unresolved |
| model fantasy batch | hypothetical observations used for conditioning |
| one-shot auxiliary rows | acquisition-internal decision variables for KG/lookahead |

これらをshape上で相互に読み替えません。

## 10. Compatibility boundaries

- standard exact GPはq-batch、pending-aware acquisition、`fantasize()`を利用できます。
- MultiTask fantasy inputsではtask identityを正しく指定する必要があります。
- Kronecker MultiTask GPは現在`supports_fantasize=False`です。private GPyTorch cache workaroundは採用しません。
- empirical tree ensembleやNGBoostへExactGP fantasy semanticsを強制しません。
- multi-fidelity pending pointのfidelityは実際に走っているevaluation contextです。
- qMFKG/qKG内部fantasyはreal unresolved `X_pending`とは別です。
- mixed one-shot qKG/qMFKGにはaugmented batch全体のcategorical assignment semanticsがあります。

## 11. No custom async scheduler

robotorchanはscheduler stateを別抽象化しません。async BOは、completed observations、unresolved `X_pending`、pending-aware acquisition、candidate optimizerのcompositionとして表現します。

## 12. Related documentation

- `docs-internal/contracts/optimization/batch-async-fantasization.md`
- `docs/theory/acquisition/04_batch_noisy.md`
- `docs/theory/optimization/08_batch_one_shot.md`
- `39_acquisition_optimization.ipynb`

Batch、Async、Fantasizationは同じ「複数点」に見えても、real candidate、pending context、hypothetical conditioningという異なる意味を持ちます。

## 13. robotorchan：Pendingを考慮したjoint EPIG

`qConditionalExpectedPredictiveInformationGain` は、すでに実験中の候補を考慮しながら、**新たに提案する2候補がターゲット領域の予測へもたらす追加情報量**を評価します。

`target_X` は予測を改善したい領域であり、`X_pending` はまだ結果がない実験中の入力です。ここでは学習済みのGPをそのまま使います。


### EPIGとqNEIの目的の違い

`qLogNoisyExpectedImprovement` は**良い目的関数値の発見**を重視するBO獲得関数です。一方、`qConditionalExpectedPredictiveInformationGain` は、**ターゲット領域の予測に関する情報獲得**を重視するActive Learning獲得関数です。

どちらもpending点を考慮できますが、目的が異なるため獲得関数値の絶対値を直接比較しません。

In [ ]:
from robotorchan.acquisition import qConditionalExpectedPredictiveInformationGain

epig_model = model
target_X = torch.linspace(0.05, 0.95, 21, dtype=torch.double).unsqueeze(-1)
epig_acq = qConditionalExpectedPredictiveInformationGain(
    epig_model, target_X, observation_noise=1e-3
)
epig_acq.set_X_pending(torch.tensor([[0.32]], dtype=torch.double))
epig_bounds = torch.tensor([[0.0], [1.0]], dtype=torch.double)
new_batch, incremental_information = optimize_acqf(
    acq_function=epig_acq,
    bounds=epig_bounds,
    q=2,
    num_restarts=2,
    raw_samples=16,
    options={"maxiter": 30},
)
print("new batch:", new_batch)
print("incremental joint information gain:", incremental_information.item())
assert new_batch.shape == (2, 1)
assert torch.isfinite(incremental_information)


### Pending状態を変更した場合

`set_X_pending(None)` で未完了候補の条件を外し、同じ新規候補に対する情報量を評価します。**Pendingを解除する操作は観測が完了したことを意味しません**。実験が完了したら、測定値を取得してGPを再学習する必要があります。


### Pendingを外した情報量の評価

次のセルでは同じ2候補に対して、pendingを考慮した場合と考慮しない場合を区別します。pendingを解除しても観測済みデータは増えません。

実験が完了した後のモデル更新は、先ほどの `updated_model.fit()` の例に従ってください。

In [ ]:
epig_acq.set_X_pending(None)
unconditional_information = epig_acq(new_batch.unsqueeze(0))
print("joint information gain without pending evaluations:", unconditional_information.item())
assert torch.isfinite(unconditional_information).all()


## 14. 実務でのAPI選択

| 状況 | 既存API | 重要な点 |
|---|---|---|
| 複数点をまとめて提案 | `qLogNoisyExpectedImprovement` + `optimize_acqf(q=2)` | joint batch |
| 未完了の実験を考慮 | `X_pending` | 観測値はまだない |
| 仮想観測で条件付け | `model.fantasize()` | fantasy batchは実験数ではない |
| 実験完了後の更新 | `SingleTaskGP.fit()` | 完了したX・Yのみ追加 |
| Pendingを考慮した情報獲得 | `qConditionalExpectedPredictiveInformationGain` | ターゲット領域とpending点を区別 |

**重要：** 実験が完了していない点を観測済みとして学習させないこと、実験装置へ送る候補とfantasy補助変数を混同しないことが、非同期運用の基本です。